# UrbanCart — Milestone 1: Predicting Next-Month Customer Spend

This notebook solves the finance team's regression problem. It:
1. loads and checks the historical customer data,
2. splits the data before training,
3. trains two suitable regression approaches,
4. evaluates them on the held-out test set using MAE, RMSE, and R²,
5. interprets the results and makes a recommendation based on the evidence.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('milestone-1-customer-spend.csv')
df.head()


,MonthsActive,AvgOrderValue,NumOrdersLastQuarter,Region,NextMonthSpend
0,6,25.70,7,South,88.51
1,47,85.81,5,East,137.62
2,40,87.63,7,East,148.99
3,27,45.16,4,East,101.25
4,26,67.26,7,East,136.76


In [2]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
display(df.describe(include='all'))


(20000, 5)
MonthsActive              int64
AvgOrderValue           float64
NumOrdersLastQuarter      int64
Region                      str
NextMonthSpend          float64
dtype: object
MonthsActive            0
AvgOrderValue           0
NumOrdersLastQuarter    0
Region                  0
NextMonthSpend          0
dtype: int64


,MonthsActive,AvgOrderValue,NumOrdersLastQuarter,Region,NextMonthSpend
count,20000.000000,20000.000000,20000.000000,20000,20000.000000
unique,NaN,NaN,NaN,4,NaN
top,NaN,NaN,NaN,North,NaN
freq,NaN,NaN,NaN,6091,NaN
mean,30.291200,59.587698,5.009200,NaN,114.011969
std,17.288798,24.875161,1.999454,NaN,29.594273
min,1.000000,11.530000,1.000000,NaN,16.040000
25%,15.000000,41.910000,4.000000,NaN,93.630000
50%,30.000000,54.875000,5.000000,NaN,111.510000
75%,45.000000,71.920000,6.000000,NaN,131.350000


## 1. Define the prediction task

`NextMonthSpend` is a continuous numeric target, so this is a **supervised regression** problem.

The predictors are `MonthsActive`, `AvgOrderValue`, `NumOrdersLastQuarter`, and categorical `Region`.


In [3]:
X = df.drop(columns='NextMonthSpend')
y = df['NextMonthSpend']

# Split BEFORE fitting either model or any learned preprocessing.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))


Training rows: 16000
Test rows: 4000


## 2. Model 1 — Linear Regression

Linear regression is a sensible first model because it is simple, interpretable, and provides a useful baseline for a continuous spending target. `Region` is one-hot encoded. Numeric variables are standardized inside the pipeline.


In [4]:
num_cols = ['MonthsActive', 'AvgOrderValue', 'NumOrdersLastQuarter']
cat_cols = ['Region']

preprocess_lr = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
])

linear_model = Pipeline([
    ('preprocess', preprocess_lr),
    ('model', LinearRegression())
])

linear_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['MonthsActive','AvgOrderValue','NumOrdersLastQuarter','Region']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concaten

## 3. Model 2 — Random Forest Regression

A random forest can capture nonlinear relationships and interactions that a straight-line model cannot. The preprocessing and model are kept in one pipeline so the test set is not used to fit preprocessing.


In [5]:
preprocess_rf = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

forest_model = Pipeline([
    ('preprocess', preprocess_rf),
    ('model', RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

forest_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['MonthsActive','AvgOrderValue','NumOrdersLastQuarter','Region']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concaten

In [6]:
# Evaluate only on the held-out test set.
pred_lr = linear_model.predict(X_test)
pred_rf = forest_model.predict(X_test)

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [mean_absolute_error(y_test, pred_lr), mean_absolute_error(y_test, pred_rf)],
    'RMSE': [mean_squared_error(y_test, pred_lr) ** 0.5, mean_squared_error(y_test, pred_rf) ** 0.5],
    'R2': [r2_score(y_test, pred_lr), r2_score(y_test, pred_rf)]
})

results.round(3)


,Model,MAE,RMSE,R2
0,Linear Regression,9.846,12.330,0.830
1,Random Forest,10.485,13.077,0.809


## 4. Interpreting the evaluation

- **MAE** is the average absolute prediction error, in the same currency units as spending. Lower is better.
- **RMSE** is also in spending units, but penalizes large errors more heavily. Lower is better.
- **R²** measures the proportion of variation in next-month spending explained by the model relative to predicting the training-set mean. Higher is better.

For this dataset, the held-out test results are approximately:

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Linear Regression | 9.846 | 12.330 | 0.830 |
| Random Forest | 10.487 | 13.077 | 0.809 |

Linear regression therefore has the lower error on both MAE and RMSE and the higher R² on this held-out test set.


In [7]:
# Optional: inspect linear-model coefficients in original units for interpretation.
preprocess_lr_raw = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
])

interpretable_lr = Pipeline([
    ('preprocess', preprocess_lr_raw),
    ('model', LinearRegression())
])

interpretable_lr.fit(X_train, y_train)

feature_names = interpretable_lr.named_steps['preprocess'].get_feature_names_out()
coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': interpretable_lr.named_steps['model'].coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('Intercept:', round(interpretable_lr.named_steps['model'].intercept_, 3))
coefficients.round(3)


Intercept: 22.873


,Feature,Coefficient
3,cat__Region_North,10.016
5,cat__Region_West,-9.625
2,num__NumOrdersLastQuarter,5.935
4,cat__Region_South,-4.892
1,num__AvgOrderValue,0.899
0,num__MonthsActive,0.265


## 5. Recommendation

**Recommendation: use Linear Regression for this milestone.**

The reason is evidence from the held-out test set: its MAE is about 9.85 versus 10.49 for the random forest, its RMSE is about 12.33 versus 13.08, and its R² is about 0.830 versus 0.809. In practical terms, the linear model's predictions are typically about 9.85 spending units away from actual next-month spend on average, while the random forest's average absolute error is about 10.49.

The tradeoff is that random forests are more flexible and can represent nonlinearities/interactions, but that flexibility did not improve test-set performance here. The linear model is also easier to explain to finance stakeholders.

The coefficients describe associations in the fitted model; they should not be interpreted as causal effects.
